# STEP 7 — Machine Learning Models

## Objective

The objective of this step is to train and evaluate machine learning models for population prediction.

### Models

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

### Evaluation Metrics

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² Score

A time-based training and testing split is used to avoid future data leakage.

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv(
    "../data/processed/Global_Population_Feature_Engineered.csv"
)

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (13858, 10)


,Country Name,Country Code,Indicator Name,Indicator Code,Year,Population,Previous_Year_Population,Population_Growth_Percentage,Population_Lag_2,Population_Rolling_Avg_3
0,Aruba,ABW,"Population, total",SP.POP.TOTL,1962,56320.0,55578.0,1.335061,54922.0,55606.666667
1,Aruba,ABW,"Population, total",SP.POP.TOTL,1963,57002.0,56320.0,1.210938,55578.0,56300.000000
2,Aruba,ABW,"Population, total",SP.POP.TOTL,1964,57619.0,57002.0,1.082418,56320.0,56980.333333
3,Aruba,ABW,"Population, total",SP.POP.TOTL,1965,58190.0,57619.0,0.990993,57002.0,57603.666667
4,Aruba,ABW,"Population, total",SP.POP.TOTL,1966,58694.0,58190.0,0.866128,57619.0,58167.666667


In [3]:
df = df.sort_values(
    ["Country Code", "Year"]
).reset_index(drop=True)

In [4]:
df["Previous_3Year_Avg"] = (
    df.groupby("Country Code")["Population"]
      .transform(
          lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
      )
)

In [5]:
ml_df = df.dropna(
    subset=[
        "Previous_Year_Population",
        "Population_Lag_2",
        "Previous_3Year_Avg"
    ]
).copy()

print("ML Dataset Shape:", ml_df.shape)

ML Dataset Shape: (13641, 11)


In [6]:
features = [
    "Year",
    "Previous_Year_Population",
    "Population_Lag_2",
    "Previous_3Year_Avg"
]

target = "Population"

X = ml_df[features]
y = ml_df[target]

print("Features:", features)
print("Target:", target)

Features: ['Year', 'Previous_Year_Population', 'Population_Lag_2', 'Previous_3Year_Avg']
Target: Population


In [7]:
train_df = ml_df[ml_df["Year"] <= 2020].copy()
test_df = ml_df[ml_df["Year"] > 2020].copy()

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

print(
    "Training Years:",
    train_df["Year"].min(),
    "-",
    train_df["Year"].max()
)

print(
    "Testing Years:",
    test_df["Year"].min(),
    "-",
    test_df["Year"].max()
)

Training Data: (12556, 4)
Testing Data: (1085, 4)
Training Years: 1963 - 2020
Testing Years: 2021 - 2025


Linear Regression

In [8]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

print("Linear Regression training completed.")

Linear Regression training completed.


In [9]:
linear_mae = mean_absolute_error(
    y_test,
    linear_predictions
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_predictions
    )
)

linear_r2 = r2_score(
    y_test,
    linear_predictions
)

print("Linear Regression")
print("-----------------------")
print("MAE :", linear_mae)
print("RMSE:", linear_rmse)
print("R²  :", linear_r2)

Linear Regression
-----------------------
MAE : 58201.18587794908
RMSE: 230328.97360872736
R²  : 0.9999973636047818


Decision Tree Regressor

In [10]:
decision_tree_model = DecisionTreeRegressor(
    max_depth=10,
    random_state=42
)

decision_tree_model.fit(X_train, y_train)

decision_tree_predictions = decision_tree_model.predict(X_test)

print("Decision Tree training completed.")

Decision Tree training completed.


In [11]:
decision_tree_mae = mean_absolute_error(
    y_test,
    decision_tree_predictions
)

decision_tree_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        decision_tree_predictions
    )
)

decision_tree_r2 = r2_score(
    y_test,
    decision_tree_predictions
)

print("Decision Tree Regressor")
print("-----------------------")
print("MAE :", decision_tree_mae)
print("RMSE:", decision_tree_rmse)
print("R²  :", decision_tree_r2)

Decision Tree Regressor
-----------------------
MAE : 425897.44499532465
RMSE: 2375325.412617307
R²  : 0.9997196118850353


Random Forest

In [12]:
random_forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

random_forest_predictions = random_forest_model.predict(X_test)

print("Random Forest training completed.")

Random Forest training completed.


In [13]:
random_forest_mae = mean_absolute_error(
    y_test,
    random_forest_predictions
)

random_forest_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        random_forest_predictions
    )
)

random_forest_r2 = r2_score(
    y_test,
    random_forest_predictions
)

print("Random Forest Regressor")
print("-----------------------")
print("MAE :", random_forest_mae)
print("RMSE:", random_forest_rmse)
print("R²  :", random_forest_r2)

Random Forest Regressor
-----------------------
MAE : 397226.79664116155
RMSE: 2492262.326676782
R²  : 0.9996913254127756


Gradient Boosting

In [14]:
gradient_boosting_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gradient_boosting_model.fit(X_train, y_train)

gradient_boosting_predictions = (
    gradient_boosting_model.predict(X_test)
)

print("Gradient Boosting training completed.")

Gradient Boosting training completed.


In [15]:
gradient_boosting_mae = mean_absolute_error(
    y_test,
    gradient_boosting_predictions
)

gradient_boosting_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        gradient_boosting_predictions
    )
)

gradient_boosting_r2 = r2_score(
    y_test,
    gradient_boosting_predictions
)

print("Gradient Boosting Regressor")
print("---------------------------")
print("MAE :", gradient_boosting_mae)
print("RMSE:", gradient_boosting_rmse)
print("R²  :", gradient_boosting_r2)

Gradient Boosting Regressor
---------------------------
MAE : 736436.5393729819
RMSE: 3672312.3091319446
R²  : 0.9993298182651322


In [16]:
model_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        linear_mae,
        decision_tree_mae,
        random_forest_mae,
        gradient_boosting_mae
    ],
    "RMSE": [
        linear_rmse,
        decision_tree_rmse,
        random_forest_rmse,
        gradient_boosting_rmse
    ],
    "R2": [
        linear_r2,
        decision_tree_r2,
        random_forest_r2,
        gradient_boosting_r2
    ]
})

model_results

,Model,MAE,RMSE,R2
0,Linear Regression,58201.185878,2.303290e+05,0.999997
1,Decision Tree,425897.444995,2.375325e+06,0.999720
2,Random Forest,397226.796641,2.492262e+06,0.999691
3,Gradient Boosting,736436.539373,3.672312e+06,0.999330


In [17]:
best_model_row = model_results.loc[
    model_results["MAE"].idxmin()
]

print("Best Model:", best_model_row["Model"])
print("MAE:", best_model_row["MAE"])
print("RMSE:", best_model_row["RMSE"])
print("R²:", best_model_row["R2"])

Best Model: Linear Regression
MAE: 58201.18587794908
RMSE: 230328.97360872736
R²: 0.9999973636047818


Actual vs Predicted Data

In [18]:
prediction_results = test_df[
    ["Country Name", "Country Code", "Year", "Population"]
].copy()

prediction_results["Predicted_Population"] = linear_predictions

prediction_results["Absolute_Error"] = (
    prediction_results["Population"]
    - prediction_results["Predicted_Population"]
).abs()

prediction_results.head(10)

,Country Name,Country Code,Year,Population,Predicted_Population,Absolute_Error
59,Aruba,ABW,2021,107700.0,1.098804e+05,2180.435598
60,Aruba,ABW,2022,107310.0,1.087512e+05,1441.175472
61,Aruba,ABW,2023,107359.0,1.089005e+05,1541.489999
62,Aruba,ABW,2024,107995.0,1.093639e+05,1368.903296
63,Aruba,ABW,2025,108785.0,1.105795e+05,1794.462459
123,Afghanistan,AFG,2021,40000412.0,4.029041e+07,289997.480280
124,Afghanistan,AFG,2022,40578842.0,4.090829e+07,329443.679422
125,Afghanistan,AFG,2023,41454761.0,4.112628e+07,328476.219151
126,Afghanistan,AFG,2024,42647492.0,4.235324e+07,294249.645898
127,Afghanistan,AFG,2025,43844111.0,4.386549e+07,21383.233973


In [19]:
model_results.to_csv(
    "../outputs/tables/model_comparison.csv",
    index=False
)

prediction_results.to_csv(
    "../outputs/tables/linear_regression_predictions.csv",
    index=False
)

print("Model comparison saved.")
print("Prediction results saved.")

Model comparison saved.
Prediction results saved.
